---
title: "Capstone: Evidence Brief Agent"
draft: true
categories: [agents, workflows, langgraph, capstone]
---


The capstone integrates intake, planning, parallel retrieval and extraction, fan-in, reconciliation, drafting, review, verification, and export. The result is both a Markdown brief and a machine-readable record whose claims can be checked without reading model logs.

## The deterministic happy path

The canonical question asks whether AtlasVector should serve regulated EU documents. Expected simulated behavior is explicit: three branches, a visible residency contradiction, a limited-pilot recommendation, approval, citation verification, and terminal status `complete`.


In [1]:
from IPython.display import Markdown, display
from evidence_brief.workflow import run_fixture

state = run_fixture("conflict-01")
display(Markdown(state["artifact"]["markdown"]))
print({
    "status": state["status"],
    "recommendation": state["artifact"]["recommendation"],
    "branches": sorted({row["task_id"] for row in state["branch_results"]}),
    "contradictions": [row["id"] for row in state["contradictions"]],
    "events": state["events"],
})
assert state["status"] == "complete"
assert state["artifact"]["recommendation"] == "pilot_only"
assert len(state["artifact"]["citations"]) == len(state["claims"])


## Recommendation

**pilot_only** for: Should Northstar adopt AtlasVector for a regulated EU document-search service?

## Evidence

- An independent audit verified audit-log export. [independent-audit-2026#52-185]
- The audit found that EU residency was not available. [independent-audit-2026#52-185]
- Recall and p95 latency passed pilot thresholds. [internal-benchmark-2026#116-193]
- Indexing recovered after a worker restart. [operations-review-2026#69-183]
- A capacity run is recommended before production. [operations-review-2026#69-183]
- Regulated production requires verified residency. [regulatory-policy-2026#0-176]
- An outdated vendor benchmark reports sub-100ms p95. [vendor-benchmark-2024#0-93]
- AtlasVector exposes an exportable audit log. [vendor-security-2026#114-194]
- The vendor guide says EU residency is available. [vendor-security-2026#114-194]

## Contradictions and uncertainty

- EU data residency: Treat residency as unverified until an independent deployment test resolves it.

{'status': 'complete', 'recommendation': 'pilot_only', 'branches': ['operations', 'performance', 'security'], 'contradictions': ['residency-conflict'], 'events': ['intake:accepted', 'plan:valid', 'collect:security', 'collect:performance', 'collect:operations', 'join:complete', 'reconcile:complete', 'draft:created', 'review:approved', 'verify:passed', 'export:complete']}


The recommendation is conservative for a concrete reason: the vendor residency claim conflicts with the independent deployment audit, while internal policy requires verification. The contradiction is retained rather than silently resolved in favor of the desired answer.

## Controlled failures and scorecard

Every injected failure either recovers through a named route or terminates with an explicit reason. The process-restart case is exercised against SQLite in Chapter 07 and the project test suite.


In [2]:
from evidence_brief.evaluation import evaluate_suite
from evidence_brief.schemas import FaultPlan

failure_runs = {
    "malformed output": run_fixture("conflict-01", faults=FaultPlan(malformed_plan=True)),
    "retrieval timeout": run_fixture("conflict-01", faults=FaultPlan(transient_retrieval_failures=1)),
    "missing branch": run_fixture("conflict-01", faults=FaultPlan(missing_task_id="operations")),
    "review rejection": run_fixture(
        "conflict-01",
        faults=FaultPlan(revision_budget_exhausted=True),
        review_decision={"action": "reject", "reason": "unresolved risk"},
    ),
    "policy violation": run_fixture("conflict-01", faults=FaultPlan(policy_violation=True)),
}
for name, run in failure_runs.items():
    print(name, "->", run.get("status"), "/", run.get("terminal_reason"))

reports = {variant: evaluate_suite(variant).means for variant in ("full", "skill_baseline", "no_review", "no_reconciliation")}
print(reports)
assert len(evaluate_suite("full").rows) == 12
assert failure_runs["missing branch"]["status"] == "failed"
assert failure_runs["review rejection"]["terminal_reason"] == "revision budget exhausted"


malformed output -> complete / completion contract satisfied
retrieval timeout -> complete / completion contract satisfied
missing branch -> failed / missing branches: operations
review rejection -> failed / revision budget exhausted
policy violation -> failed / source policy violation


{'full': {'recommendation_correct': 1.0, 'evidence_coverage': 1.0, 'claim_support': 1.0, 'provenance_completeness': 1.0, 'contradiction_detection': 1.0, 'legal_transitions': 1.0, 'bounded_termination': 1.0, 'review_compliance': 1.0, 'citation_correctness': 1.0, 'resume_correctness': 1.0}, 'skill_baseline': {'recommendation_correct': 0.75, 'evidence_coverage': 0.75, 'claim_support': 0.75, 'provenance_completeness': 0.5, 'contradiction_detection': 0.833, 'legal_transitions': 0.5, 'bounded_termination': 1.0, 'review_compliance': 0.0, 'citation_correctness': 0.75, 'resume_correctness': 0.0}, 'no_review': {'recommendation_correct': 1.0, 'evidence_coverage': 1.0, 'claim_support': 1.0, 'provenance_completeness': 1.0, 'contradiction_detection': 1.0, 'legal_transitions': 1.0, 'bounded_termination': 1.0, 'review_compliance': 0.0, 'citation_correctness': 1.0, 'resume_correctness': 1.0}, 'no_reconciliation': {'recommendation_correct': 1.0, 'evidence_coverage': 1.0, 'claim_support': 1.0, 'provenanc

## Optional live adapter

Live calls are an integration extension, not evidence for the course. The guarded cell below cannot run unless both an API key and an explicit switch are present; stored outputs always come from the scripted adapter.


In [3]:
import os

RUN_LIVE = False
if RUN_LIVE and os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    from evidence_brief.adapters import OpenAIModelAdapter
    from evidence_brief.schemas import BriefRequest
    live_adapter = OpenAIModelAdapter(OpenAI())
    print(live_adapter.plan(BriefRequest.model_validate(state["request"])))
else:
    print("Live adapter skipped; deterministic fixture outputs remain canonical.")


Live adapter skipped; deterministic fixture outputs remain canonical.


## Decision memo

Reusable source-handling and writing guidance belongs in **skills**. Retrieval, offset validation, citation rendering, and export belong in **deterministic tools**. Pause/resume, fan-out/fan-in, bounded recovery, checkpointing, and review belong in **graph control flow**. The citation renderer, source catalog, and one-shot formatting steps gain nothing from being graph nodes; ordinary functions remain the clearer implementation.
